# Income Pipeline, Part 3: API (Census Population by ZIP Code)

**Goal:** download 2010 census population by ZIP code through the Kaggle API, convert it to JSON, and reduce it to one population figure per ZIP code.

**Steps:** authenticate with a key stored in a JSON file and download the dataset (shown, but only run when the key is present), convert the CSV to JSON, work out how the file is structured, and keep only the total row for each ZIP code.

**Result:** the file lists every ZIP code three ways (by gender and age band, by gender, and in total), so adding every row gives 937 million people. Using only the total rows gives 312,444,427, the 2010 population of the U.S. and Puerto Rico, across 33,119 ZIP codes (50.9% female). Saved as `Data/Census_API.csv`.

**Tools:** Python (Kaggle API, json, pandas)

**Context:** master's coursework (DSC540 term project), cleaned up and corrected for this portfolio. This is part of a four-notebook pipeline: 1 flat file, 2 web scraping, 3 API, 4 merge and visualize.

## 1. Download the data with the Kaggle API
**About the API key:** these cells show the full Kaggle API workflow: reading the key from a JSON file, authenticating, and downloading the dataset. The key file is not published with this project, so the API cells only run when `Kaggle.json` is present. When the notebook was run with the key, the download printed `Dataset URL: https://www.kaggle.com/datasets/census/us-population-by-zip-code` and saved `Data/population_by_zip_2010.csv`, which is included here so the rest of the notebook runs without a key.

In [1]:
import os
import json
import pandas as pd

# the kaggle package checks for credentials as soon as it is imported, so the import is wrapped
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
except (OSError, ImportError) as e:
    print('Kaggle API not available on this computer:', e)

HAS_KEY = os.path.exists('Kaggle.json')
print('Kaggle.json found: API steps will run' if HAS_KEY else 'Kaggle.json not found: skipping the API download, using the saved CSV')

Kaggle.json not found: skipping the API download, using the saved CSV


In [2]:
if HAS_KEY:
    # read the API key from the JSON file and put it where the Kaggle library expects it
    with open('Kaggle.json') as f:
        keys = json.load(f)
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        import shutil
        shutil.copy('Kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.environ['KAGGLE_CONFIG_DIR'] = os.path.expanduser('~/.kaggle')

    # authenticate and download the census dataset
    api = KaggleApi()
    api.authenticate()
    api.dataset_download_files('census/us-population-by-zip-code', path='Data', unzip=True)

## 2. Convert the CSV to JSON
The project required a JSON source. Kaggle only offers this dataset as CSV, so the CSV is converted to JSON (one record per line). The JSON is large (178 MB), so it is only rebuilt when missing.

In [3]:
census = pd.read_csv('Data/population_by_zip_2010.csv')
json_file = 'Data/population_by_zip_2010.json'
if not os.path.exists(json_file):
    census.to_json(json_file, orient='records', lines=True, index=False)
census = pd.read_json(json_file, orient='records', lines=True)   # read back from JSON for the rest of the notebook
print(census.shape)
census.head()

(1622831, 6)


,population,minimum_age,maximum_age,gender,zipcode,geo_id
0,50,30.0,34.0,female,61747,8600000US61747
1,5,85.0,NaN,male,64120,8600000US64120
2,1389,30.0,34.0,male,95117,8600000US95117
3,231,60.0,61.0,female,74074,8600000US74074
4,56,0.0,4.0,female,58042,8600000US58042


## 3. Understand the structure: each ZIP code appears three ways
For every ZIP code the file has (1) one row per gender and age band, (2) a total for each gender (age columns blank), and (3) an overall total (gender and age blank). Adding every row would count each person three times, so only the right kind of row is used for each purpose.

In [4]:
age_blank = census['minimum_age'].isna() & census['maximum_age'].isna()
overall = census[age_blank & census['gender'].isna()]
by_gender = census[age_blank & census['gender'].notna()]
print(f"Sum of every row:        {census['population'].sum():,}  (counts people three times)")
print(f"Sum of overall totals:   {overall['population'].sum():,}  (2010 population of the U.S. ZIP codes and Puerto Rico)")
print(f"Sum of gender totals:    {by_gender['population'].sum():,}")
print(f"ZIP codes: {overall['zipcode'].nunique():,}")

Sum of every row:        937,333,281  (counts people three times)
Sum of overall totals:   312,444,427  (2010 population of the U.S. ZIP codes and Puerto Rico)
Sum of gender totals:    312,444,427
ZIP codes: 33,119


In [5]:
# population split by gender, from the gender total rows
gender_split = by_gender.groupby('gender')['population'].sum()
print((gender_split / gender_split.sum() * 100).round(1).to_string())

gender
female    50.9
male      49.1


## 4. Save one row per ZIP code

In [6]:
census_zip = (overall[['zipcode', 'geo_id', 'population']]
              .rename(columns={'zipcode': 'ZipCode', 'population': 'PopulationCensus'}))
census_zip['ZipCode'] = census_zip['ZipCode'].astype(str).str.zfill(5)
print('Duplicate ZIP codes:', census_zip['ZipCode'].duplicated().sum())
census_zip.to_csv('Data/Census_API.csv', index=False)
gender_split.rename('PopulationCensus').to_csv('Data/Census_gender_totals.csv')
print(f'Saved Data/Census_API.csv: {len(census_zip):,} rows (one per ZIP code)')

Duplicate ZIP codes: 0


Saved Data/Census_API.csv: 33,119 rows (one per ZIP code)
